In [5]:
import numpy as np
import pandas as pd

import geopandas as gpd

from datetime import datetime

import os
import geopandas as gpd


import pymannkendall as mk
from scipy import stats



workpath=r'E:\drought'
dataname='SPEI'
data_path=os.path.join(workpath,'data','SPEI','SPEI')
stinfo=pd.read_excel(os.path.join(data_path,'站点编号及信息_选定_1969-2022.xlsx'))
stinfo['station'] = stinfo['station'].apply(str)
station_list=list(stinfo['station'])
lat_list=list(stinfo['lat'])
lon_list=list(stinfo['lon'])  
st_sum=len(station_list)
startyear,endyear=1982,2022

In [4]:
#合并站点SPEI
for dataname in ['SPEI','clmSPEI']:
    if dataname=='SPEI':speipath='SPEI_PM'
    if dataname=='clmSPEI':speipath='SPEI_TH'
    SPEI_Data_path=os.path.join(workpath,'data','SPEI','SPEI',speipath)
    SPEI_Merge_Path=os.path.join(SPEI_Data_path,'month_merge')    
    
    fnpath=os.path.join(SPEI_Data_path,'month')    
    dfs=pd.DataFrame()
    print('合并站点:工作目录'+SPEI_Data_path+',......')
    for i in range(st_sum):    
        st_no=stinfo['station'][i]
        fn=os.path.join(fnpath,str(st_no)+".csv")
        df=pd.read_csv(fn)
        df['station']=st_no
        dfs=pd.concat([dfs,df])

    dfs=dfs[(dfs['year']>=startyear)&(dfs['year']<=endyear)]
    outfn=os.path.join(SPEI_Merge_Path,dataname+'_month_'+str(startyear)+'_'+str(endyear)+'.csv')
    dfs.to_csv(outfn)
    print('this is ok',outfn,datetime.now())#step 01 合并所有站点

合并站点:工作目录E:\drought\data\SPEI\SPEI\SPEI_PM,......
this is ok E:\drought\data\SPEI\SPEI\SPEI_PM\month_merge\SPEI_month_1982_2022.csv 2023-09-21 00:04:18.735609
合并站点:工作目录E:\drought\data\SPEI\SPEI\SPEI_TH,......
this is ok E:\drought\data\SPEI\SPEI\SPEI_TH\month_merge\clmSPEI_month_1982_2022.csv 2023-09-21 00:04:43.469256


In [6]:
#将SPEI csv格式转shp 接下来进行格点化
print('提取1-12月尺度的干旱指数,并转换为shp格式')
for dataname in ['SPEI','clmSPEI']:
    if dataname=='SPEI':speipath='SPEI_PM'
    if dataname=='clmSPEI':speipath='SPEI_TH'
    SPEI_Data_path=os.path.join(workpath,'data','SPEI','SPEI',speipath)
    SPEI_Merge_Path=os.path.join(SPEI_Data_path,'month_merge')    
    SPEI_Shp_Path=os.path.join(SPEI_Data_path,'month_shp')   
    
    fn=os.path.join(SPEI_Merge_Path,dataname+'_month_'+str(startyear)+'_'+str(endyear)+'.csv')
    dfs=pd.read_csv(fn)   
    dfs=dfs.iloc[:,2:]
    #初如化列名
    colnames=[]
    for iy in range(startyear,endyear+1):
        for im in range(1,13):
            colnames.append('S'+str(iy)+'{:02d}'.format(im))

    t_list=list(range(1,13))    
    for t in t_list:#时间尺度
        sel_d=dataname+str(t)+'_Month'
        #print(sel_d)
        data = dfs.pivot_table(index='station',columns=['year','month'],values=sel_d,aggfunc='mean')
        data.columns=colnames   

        stinfo['station']=stinfo['station'].astype(int)
        data=data.merge(stinfo,how='left',on='station')[['station','lon','lat']+colnames]
        data.iloc[:,1:]=data.iloc[:,1:].astype(float)

        csvfn=os.path.join(SPEI_Merge_Path,sel_d+'_'+str(startyear)+'_'+str(endyear)+'.csv')
        shpfn=os.path.join(SPEI_Shp_Path,sel_d  +'_'+str(startyear)+'_'+str(endyear)+'.shp')

        data.to_csv(csvfn)        
        gdf = gpd.GeoDataFrame(data, geometry=gpd.points_from_xy(data.lon, data.lat))
        gdf.to_file(shpfn, driver='ESRI Shapefile')
        print(shpfn)
    print('it is ok',datetime.now()) 


提取1-12月尺度的干旱指数,并转换为shp格式
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI1_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI2_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI3_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI4_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI5_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI6_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI7_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI8_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI9_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI10_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI11_Month_1982_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_PM\month_shp\SPEI12_Month_1982_2022.shp
it is ok 2023-09-21 22:37:22.675297
E:\drought\data\SPEI\SPEI\SPEI_TH\month_shp\clmSPEI1_Month_1982_2022.shp
E:\drought\data\SPEI\SP

In [40]:
#站点SPEI变化趋势 计算变化趋势
Methods=['MK','Line']
selMethod='MK'
startyear,endyear=2001,2022


season_ID=['S1','S2','S3','S4','GS','Y']
Seasons=['春季','夏季','秋季','冬季','生长季','全年']
spei_season_ms=[5,8,11,2,10,12]

for dataname in ['SPEI','clmSPEI']:
    if dataname=='SPEI':speipath='SPEI_PM'
    if dataname=='clmSPEI':speipath='SPEI_TH'
    SPEI_Data_path=os.path.join(workpath,'data','SPEI','SPEI',speipath)
    SPEI_Merge_Path=os.path.join(SPEI_Data_path,'month_merge')    
    SPEI_Shp_Path=os.path.join(SPEI_Data_path,'month_shp')   
    SPEI_TrendShp_Path=os.path.join(SPEI_Data_path,'Trend_shp') 
    
    fn=os.path.join(SPEI_Merge_Path,dataname+'_month_'+str(startyear)+'_'+str(endyear)+'.csv')
    df=pd.read_csv(fn,encoding='utf-8')
    df=df[(df['year']>=startyear)]
    outdf=stinfo[['station','lat','lon']].copy()    
    outfn=os.path.join(SPEI_TrendShp_Path,
                       '_'.join([dataname,'trend',str(startyear),str(endyear)]))
    for k in range(len(season_ID)):
        arr_avg,arr_slope,arr_p=[],[],[]
        sel_m=spei_season_ms[k]
        sel_s=season_ID[k]
        tscale=3
        if sel_s=='GS':tscale=7
        if sel_s=='Y':tscale=12
        sel_fld=dataname+str(tscale)+'_Month'
        for i in range(st_sum):  
            st_n=stinfo['station'][i]
            sel_df=df[(df['station']==int(st_n))&(df['month']==sel_m)]
            Y=sel_df[sel_fld]
            X=list(range(len(Y)))
            
            if selMethod=='MK':
                trend, h, p_value, z, Tau, s, var_s, slope, intercept  = mk.original_test(Y)
            if selMethod=='Line':
                slope, intercept, r_value, p_value, std_err = stats.linregress(X,Y)
                   
            arr_avg.append(np.nanmean(Y))
            arr_slope.append(slope)
            arr_p.append(p_value)
                   
       
        outdf[sel_s+'_mean']=arr_avg    
        outdf[sel_s+'_slope']=arr_slope
        outdf[sel_s+'_p']=arr_p
    
    outdf.to_csv(outfn+'.csv')
    gdf = gpd.GeoDataFrame(outdf, geometry=gpd.points_from_xy(outdf.lon, outdf.lat))
    gdf.to_file(outfn+'.shp', driver='ESRI Shapefile')
    print(outfn+'.shp')    
print('it is ok',datetime.now())  

E:\drought\data\SPEI\SPEI\SPEI_PM\Trend_shp\SPEI_trend_2001_2022.shp
E:\drought\data\SPEI\SPEI\SPEI_TH\Trend_shp\clmSPEI_trend_2001_2022.shp
it is ok 2023-09-22 00:11:11.809191


In [28]:
season_ID=['_S1','_S2','_S3','_S4','_GS','']
spei_season_ms=[5,8,11,2,10,12]
Seasons=['春季','夏季','秋季','冬季','生长季','全年']
arr=[]
      
sel_s=season_ID[k]
sel_m=spei_season_ms[k] 
tscale=3
if sel_s=='_GS':tscale=7
if sel_s=='':tscale=12
sel_dx=dataname+str(tscale)+'_Month'
for i in range(st_sum):  
    st_n=stinfo['station'][i]
    sel_df=df[(df['month']==sel_m)&(df['station']==st_n)][['station','year','month',sel_dx]]
    Y=sel_df[sel_dx].values
    slope, intercept, r_value, p_value, std_err = stats.linregress(X,Y)
    arr.append([st_n,,slope,p_value])

array([ 0.30038471,  1.22012585,  1.27038868, -0.12382325,  0.69239409,
       -1.77801592, -0.02668377, -1.04974887,  0.05761902, -1.39224046,
       -0.4067549 ,  0.47059888,  1.25591208, -0.31672903, -0.36219631,
        1.1101496 , -0.54748784,  0.11411437,  0.89350202,  0.74227476,
        1.06523119, -0.50598508, -1.21938033, -1.65218953, -1.15663885,
       -0.48852637,  0.17733787, -1.88590928, -1.18551378, -1.68558761,
       -0.21991987, -0.46261769,  0.88745162,  0.92192785, -2.03572956,
        1.20845412,  0.71145053, -0.38249656, -0.62922293,  0.08849361,
       -0.17885353])

In [8]:
col_list=dfs.columns
col_names=['T','SPEI','Scale']
dfs['T']=dfs['year']*100+dfs['month']
dfs=dfs.groupby('T').mean().reset_index()
outdf=pd.DataFrame()
for im in range(1,13):
    sel_d='clmSPEI'+str(im)+'_Month'
    df0=dfs[['T',sel_d]]
    df0.loc[:,'Scale']=im
    df0.loc[:,'T']=list(range(1,493))
    df0.columns=col_names
    df0.reset_index()
    outdf=pd.concat([outdf,df0])
outdf.to_csv(r'G:\drought\data\SPEI_climate_indices\csv\clmSPEI_month_1982_2022_等高图.csv')

C:\Users\hongx\AppData\Local\Temp\ipykernel_8960\540498046.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df0.loc[:,'Scale']=im
C:\Users\hongx\AppData\Local\Temp\ipykernel_8960\540498046.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df0.loc[:,'Scale']=im
C:\Users\hongx\AppData\Local\Temp\ipykernel_8960\540498046.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documenta

OSError: Cannot save file into a non-existent directory: 'G:\drought\data\SPEI_climate_indices\csv'

In [85]:
print(dataname+' 干旱化趋势：year from '+str(startyear)+'-'+str(endyear))
print('显著干旱化','不显著干旱化','不显著湿润','显著湿润')
for sel_s in Seasons:
    outf_part='_'+sel_s
    if sel_s=='Y':outf_part=''
    csvfn=os.path.join(SPEI_TrendShp_Path,
                       dataname+'_trend_'+str(startyear)+'_'+str(endyear-1)+outf_part+'_'+selMethod+'.csv')
    df=pd.read_csv(csvfn)  
    fd_slope=sel_s+'_slope'
    fd_p=sel_s+'_p'
    k1=df[(df[fd_p]<0.05)&(df[fd_slope]<0)]['station'].count() #显著干旱化
    k2=df[(df[fd_p]>0.05)&(df[fd_slope]<0)]['station'].count()#不显著干旱化
    k3=df[(df[fd_p]>0.05)&(df[fd_slope]>0)]['station'].count()#不显著湿润化
    k4=df[(df[fd_p]>0.05)&(df[fd_slope]>0)]['station'].count()#显著湿润化
    print(sel_s,k1/(k1+k2+k3+k4),k2/(k1+k2+k3+k4),k3/(k1+k2+k3+k4),k4/(k1+k2+k3+k4))


SPEI 干旱化趋势：year from 1982-2022
显著干旱化 不显著干旱化 不显著湿润 显著湿润
S1 0.06164383561643835 0.5 0.2191780821917808 0.2191780821917808
S2 0.05921052631578947 0.45394736842105265 0.24342105263157895 0.24342105263157895
S3 0.5 0.46551724137931033 0.017241379310344827 0.017241379310344827
S4 0.03468208092485549 0.2254335260115607 0.3699421965317919 0.3699421965317919
GS 0.13953488372093023 0.6434108527131783 0.10852713178294573 0.10852713178294573
Y 0.25 0.6048387096774194 0.07258064516129033 0.07258064516129033


In [73]:
print('显著干旱化','不显著干旱化','不显著湿润','显著湿润')
for sel_s in Seasons:
    outf_part='_'+sel_s
    if sel_s=='Y':outf_part=''
    csvfn=os.path.join(SPEI_TrendShp_Path,
                       dataname+'_trend_'+str(startyear)+'_'+str(endyear-1)+outf_part+'_'+selMethod+'.csv')
    df=pd.read_csv(csvfn)  
    fd_slope=sel_s+'_slope'
    fd_p=sel_s+'_p'
    k1=df[(df[fd_p]<0.05)&(df[fd_slope]<0)]['station'].count() #显著干旱化
    k2=df[(df[fd_p]>0.05)&(df[fd_slope]<0)]['station'].count()#不显著干旱化
    k3=df[(df[fd_p]>0.05)&(df[fd_slope]>0)]['station'].count()#不显著湿润化
    k4=df[(df[fd_p]>0.05)&(df[fd_slope]>0)]['station'].count()#显著湿润化
    print(sel_s,k1/(k1+k2+k3+k4),k2/(k1+k2+k3+k4),k3/(k1+k2+k3+k4),k4/(k1+k2+k3+k4))


显著干旱化 不显著干旱化 不显著湿润 显著湿润
S1 0.11382113821138211 0.7560975609756098 0.06504065040650407 0.06504065040650407
S2 0.06428571428571428 0.5785714285714286 0.17857142857142858 0.17857142857142858
S3 0.5726495726495726 0.37606837606837606 0.02564102564102564 0.02564102564102564
S4 0.06666666666666667 0.32727272727272727 0.30303030303030304 0.30303030303030304
GS 0.2066115702479339 0.6942148760330579 0.049586776859504134 0.049586776859504134
Y 0.4188034188034188 0.5470085470085471 0.017094017094017096 0.017094017094017096
